# Is your copy of the dataset the published one

Every group working on decoding downloads the same public releases and writes the same sentence
about which files they used. `qbc corpus` pins the digest, so that sentence can say something a
reader can check.

Nothing is redistributed here. These are other people's datasets. The manifest records where the
publisher put the file, the sha256 of a copy fetched from there, the size, the date it was fetched,
and the citation they ask for.

In [1]:
from qb_compiler.corpus import get_corpus, list_corpora

for entry in list_corpora():
    print(f"{entry.name}")
    print(f"  {entry.publisher}, {entry.n_bytes:,} bytes, fetched {entry.fetched}")
    print(f"  {entry.doi}")
    print()

quera-surface-code
  QuEra Computing, 996,799,635 bytes, fetched 2026-05-31
  10.5281/zenodo.15685795

willow-105q-d3-d5-d7
  Google Quantum AI, 5,716,907,033 bytes, fetched 2026-05-31
  10.5281/zenodo.13273331



In [2]:
willow = get_corpus("willow-105q-d3-d5-d7")
print(willow.description)
print()
print("url    :", willow.url)
print("sha256 :", willow.sha256)
print("cite   :", willow.citation)

Surface code memory experiments at distances 3, 5 and 7 on a 105 qubit device, with detection events and observable flips per shot.

url    : https://zenodo.org/records/13273331/files/google_105Q_surface_code_d3_d5_d7.zip
sha256 : 1e8e3b4f5f35ba4fd9b4a8448473f37a7090b404fd6eff0c00188092876070dd
cite   : Google Quantum AI and Collaborators, Quantum error correction below the surface code threshold, Nature (2025). Dataset: https://doi.org/10.5281/zenodo.13273331


## A file that is not the published one

The check is a digest, so a truncated download and an edited file both fail, and they fail
differently enough to tell you which happened.

In [3]:
import tempfile
from pathlib import Path

from qb_compiler.corpus import verify_corpus_file

workspace = Path(tempfile.mkdtemp(prefix="qbc-corpus-demo-"))
pretend = workspace / "google_105Q_surface_code_d3_d5_d7.zip"
pretend.write_bytes(b"PK\x03\x04 not the real archive")

result = verify_corpus_file("willow-105q-d3-d5-d7", pretend)
print(result.status)
print(result.detail)
print()
print("expected:", result.expected_sha256)
print("actual  :", result.actual_sha256)

MISMATCH
google_105Q_surface_code_d3_d5_d7.zip does not match the pinned digest for willow-105q-d3-d5-d7; the file is 25 bytes against an expected 5,716,907,033, which usually means a truncated download. Do not benchmark against it without settling why.

expected: 1e8e3b4f5f35ba4fd9b4a8448473f37a7090b404fd6eff0c00188092876070dd
actual  : 56b8229f2a07907b0b916afd39d5ee2b53dc17d23b77616d6b0646ee0d52bdc0


In [4]:
missing = verify_corpus_file("willow-105q-d3-d5-d7", workspace / "not_downloaded_yet.zip")
print(missing.status)
print(missing.detail)

MISSING
/tmp/qbc-corpus-demo-dmofhgzv/not_downloaded_yet.zip does not exist. Fetch google_105Q_surface_code_d3_d5_d7.zip from https://zenodo.org/records/13273331/files/google_105Q_surface_code_d3_d5_d7.zip and point this at it; nothing is mirrored by this package.


## And one that is

Downloading five gigabytes inside a notebook would be rude, so this registers a small file we make
here in a manifest of its own and runs the same code path against it.

In [5]:
import hashlib
import json

import qb_compiler.corpus as corpus_module

payload = b"detection_events b8 stand-in for a notebook\n"
sample = workspace / "sample.b8"
sample.write_bytes(payload)

manifest = {
    "schema": "qb.corpus_manifest.v1",
    "corpora": [
        {
            "name": "notebook-sample",
            "description": "a file created by this notebook, so the digest can be checked offline",
            "publisher": "this notebook",
            "doi": "10.5281/zenodo.0000000",
            "url": "https://example.invalid/sample.b8",
            "filename": "sample.b8",
            "sha256": hashlib.sha256(payload).hexdigest(),
            "n_bytes": len(payload),
            "fetched": "2026-08-15",
            "citation": "nobody, a file (2026)",
            "licence": "as stated by the publisher",
        }
    ],
}
manifest_path = workspace / "manifest.json"
manifest_path.write_text(json.dumps(manifest))

real_manifest = corpus_module._MANIFEST_PATH
corpus_module._MANIFEST_PATH = manifest_path

good = verify_corpus_file("notebook-sample", sample)
print(good.status)
print(good.detail)

sample.write_bytes(payload.replace(b"stand-in", b"stand_in"))
edited = verify_corpus_file("notebook-sample", sample)
print()
print(edited.status, "after changing one character")
print(edited.detail)

corpus_module._MANIFEST_PATH = real_manifest

VERIFIED
sample.b8 matches the pinned digest for notebook-sample (44 bytes). Cite: nobody, a file (2026)

MISMATCH after changing one character
sample.b8 does not match the pinned digest for notebook-sample; the size matches, so the contents differ rather than the transfer failing. Do not benchmark against it without settling why.


Same byte count, different content, and the message says so rather than guessing at a truncated
transfer.

## Refusing rather than warning

`load_corpus` raises instead of returning a path when the digest does not match. A decoder benchmark
run on unverified bytes is worse than one that did not run, because the number it produces looks
exactly the same.

In [6]:
from qb_compiler.corpus import load_corpus

try:
    load_corpus("willow-105q-d3-d5-d7", pretend)
except ValueError as exc:
    print("refused:", str(exc)[:160])

refused: google_105Q_surface_code_d3_d5_d7.zip does not match the pinned digest for willow-105q-d3-d5-d7; the file is 25 bytes against an expected 5,716,907,033, which u


In [7]:
import shutil

shutil.rmtree(workspace)
print("temporary files removed")

temporary files removed


## What a digest does and does not tell you

It says your copy is byte for byte the copy fetched on the date in the entry. It does not say the
publisher has not re-released under the same DOI since, which happens and is legitimate. A mismatch
is something to chase, starting with the record's version history, not an accusation.

There is no `qbc corpus fetch`. The publishers already serve these files properly, and a half
implemented downloader inside a compiler package is a liability rather than a feature.